# BERTScore mismatched-paper baseline (Supplementary Note 6, Table S1)

Compares human and LLM extractions for the **same paper** with deliberately mismatched
papers from the **same field**, to calibrate what a given BERTScore F1 means in this corpus.

## Run

1. Open this notebook from the `notebooks/` folder of the repository with a Python 3.11+ kernel.
2. Install the optional dependencies once: `pip install -r requirements-full.txt`
   (or set `INSTALL_DEPENDENCIES = True` in the first code cell).
3. Choose **Run All**. The default run uses all 342 retained papers and 10 random
   reassignments per field. No API keys, Zotero access or LLM calls are needed.

The notebook reads the published extraction tables in `data/extraction/` (identical to
Supplementary Data 1). Internet access is needed only to download the public RoBERTa model
and the WordNet data on first use. CPU execution is supported; a GPU is faster.

The **primary analysis** covers all eight narrative fields and all eight whole-set
BERTScore fields. An optional, slower extension evaluates the thresholded item-matching F1
for set fields. This extension is a different metric, not ordinary BERT-F1.

The outputs of the run reported in the paper are stored in `data/bertscore_baseline/`.
`scripts/03_bertscore_baseline.py` rebuilds Table S1 from them without the model.


In [ ]:
# First use only: change to True if this Python kernel lacks these packages.
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess, sys
    requirements = [
        "bert-score==0.3.13", "transformers>=4.41,<5", "torch>=2.2",
        "numpy>=1.24,<3", "pandas>=2,<3", "scipy>=1.10",
        "matplotlib>=3.7", "nltk>=3.8", "tqdm>=4.65",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", *requirements])
    print("Installation finished. If packages were already imported, restart the kernel and Run All again.")
else:
    print("Using the packages in the selected Python kernel.")


## 1. Settings

Keep the defaults for the manuscript analysis. `RUN_MODE="quick"` is an optional
installation check using 12 papers and 2 reassignments; its outputs are clearly labeled
and must not be reported as the full analysis. Re-running with unchanged inputs and
settings reuses completed scores. Changing scoring settings creates a separate cache.

The original evaluator calls `BERTScorer(lang="en")`, whose English defaults are
`roberta-large`, layer 17, no IDF weighting, no baseline rescaling, and the slow tokenizer.
These defaults are made explicit here. MPNet was used for cosine similarity in the
original analysis and is **not** the BERTScore model.


In [ ]:
from pathlib import Path

RUN_MODE = "full"                      # "full" or "quick"
N_SHUFFLES = 10
RANDOM_SEED = 20260920
INCLUDE_PAIRWISE_SET_F1 = False          # Slower optional extension; threshold = 0.70.
BATCH_SIZE = 8                         # Reduce to 2 or 4 if GPU memory is insufficient.
SCORE_CHUNK_SIZE = 4096                 # Checkpoint after each chunk.
DEVICE = "auto"                        # "auto", "cpu", or "cuda"

# Repository root (works when the notebook is opened from notebooks/ or from the root).
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

USE_EMBEDDED_DATA = False  # inputs are read from the published CSV files below
HUMAN_CSV = REPO_ROOT / "data" / "extraction" / "human_extraction.csv"
LLM_CSV = REPO_ROOT / "data" / "extraction" / "llm_extraction.csv"
OUTPUT_ROOT = REPO_ROOT / "results" / "bertscore_baseline_run"
# SHA-256 of the published Supplementary Data 1 files used for the reported run.
PUBLISHED_INPUT_SHA256 = {
    "human": "c33fa3aa6926e05075cee9df9fc91bff78796f90e9371e4f851ff0b55bc0d49e",
    "llm": "ad5bb0122333bbd14d50cb9412279aea2f077b9e0a533bfbd4965bccdc5428e1",
}

MODEL_TYPE = "roberta-large"
MODEL_LAYERS = 17
PAIR_THRESHOLD = 0.70
EXPECTED_INPUT_PAPERS = 345
EXPECTED_ANALYZED_PAPERS = 342
HISTORICAL_MEAN_TOLERANCE = 0.001

# Fixes the evaluated sample by identity, not by CSV row position.
EXCLUSIONS = {
    "10.1007/s12040-016-0786-7": {
        "zotero_key": "4VZ2CD2X", "reason": "Original extraction refusal reported in manuscript"},
    "10.1007/s10668-024-05859-w": {
        "zotero_key": "MEE88MWP", "reason": "Original extraction refusal reported in manuscript"},
    "10.1371/journal.pone.0311816": {
        "zotero_key": "SR52IV4U", "reason": "Original unusable extraction output; excluded from the original evaluation"},
}

FREE_TEXT_FIELDS = ["Objective", "Sim. Method Advantages", "Key Findings", "Implications",
                    "Limitations", "Future Directions", "Scalability", "Novelty"]
SET_FIELDS = ["LULC Classes", "Scenarios", "Analysis Methods", "Predictors", "Data Sources",
              "Tools/Soft Used", "Validation Methods", "Validation Data Sources"]
FIELDS = FREE_TEXT_FIELDS + SET_FIELDS
SOURCE_COMMIT = "543af726f861acca9f66105e557129371f9989d6"
ANALYSIS_VERSION = "1.0.0"

assert RUN_MODE in {"full", "quick"}
assert isinstance(N_SHUFFLES, int) and N_SHUFFLES >= 2
assert BATCH_SIZE >= 1 and SCORE_CHUNK_SIZE >= 1


## 2. Provenance

The input tables are the published Supplementary Data 1 files. The original evaluation
excluded three papers (Zotero keys `4VZ2CD2X`, `MEE88MWP` and `SR52IV4U`: two extraction
refusals and one unusable output), so 342 papers are analysed. The saved means from the
original evaluation (`data/extraction/insights_extraction_summary.csv`) are loaded below
for the historical-method check.

The original package versions were not recorded. The historical-method check therefore
reports differences from the saved means; it does not assume exact historical replication.
Current package versions and the resolved model revision are saved at runtime.


In [ ]:
import csv as _csv

EMBEDDED_INPUTS = None  # inputs are read from data/extraction/ (see Settings)

# Saved means from the original evaluation, used by the historical-method check.
_HISTORICAL_METRICS = {"bert_f1", "set_text_bert_f1", "soft_f1"}
with open(REPO_ROOT / "data" / "extraction" / "insights_extraction_summary.csv", encoding="utf-8") as _f:
    HISTORICAL_SUMMARY_ROWS = [
        {"Column": r["Column"], "Type": r["Type"], "Metric": r["Metric"], "Mean": float(r["Mean"])}
        for r in _csv.DictReader(_f)
        if r["Column"] in FIELDS and r["Metric"] in _HISTORICAL_METRICS
    ]
print(len(HISTORICAL_SUMMARY_ROWS), "saved field-metric means loaded")


## 3. Import packages and audit the 342-paper sample

The join uses normalized DOI values and rejects duplicate IDs, missing IDs, unequal input
sets, and unexpected sample sizes. It does not align the files by row order. Neither the
original CSVs nor repository files are modified. Input snapshots and an exclusion table
are saved inside the analysis output folder.


In [ ]:
import base64, csv, gzip, hashlib, importlib.metadata, io, json, os, platform, re
import sqlite3, sys, time, unicodedata, zipfile
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

try:
    import torch
    import nltk
    from nltk.stem import WordNetLemmatizer
    from bert_score import BERTScorer
    from scipy.optimize import linear_sum_assignment
except ImportError as exc:
    raise ImportError(
        "A required package is missing. Set INSTALL_DEPENDENCIES=True in the first "
        "code cell, run it, restart the kernel if needed, and Run All."
    ) from exc

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def digest_json(obj):
    return sha256_bytes(json.dumps(obj, sort_keys=True, ensure_ascii=False,
                                  default=str).encode('utf-8'))

def write_json(path, obj):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(obj, indent=2, ensure_ascii=False,
                                    default=str, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def write_csv(frame, name):
    path = RUN_DIR / name
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8-sig')
    temporary.replace(path)
    return path

def normalize_doi(value):
    value = str(value).strip().lower()
    value = re.sub(r'^https?://(?:dx\.)?doi\.org/', '', value)
    value = re.sub(r'^doi\s*:\s*', '', value)
    return value.strip()

def load_csv_bytes(kind, custom_path):
    if USE_EMBEDDED_DATA:
        item = EMBEDDED_INPUTS[kind]
        raw = gzip.decompress(base64.b64decode(item['gzip_base64']))
        if sha256_bytes(raw) != item['sha256']:
            raise ValueError(f'Embedded {kind} input failed its checksum.')
        source = item['filename']
    else:
        if not custom_path.is_file():
            raise FileNotFoundError(f'Cannot find {custom_path.resolve()}. Edit the CSV paths in Settings.')
        raw = custom_path.read_bytes()
        source = custom_path.name
    frame = pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig', dtype=str, keep_default_na=False)
    frame.columns = frame.columns.str.strip()
    required = ['DOI', 'Title'] + FIELDS
    absent = [col for col in required if col not in frame]
    if absent:
        raise ValueError(f'{kind} CSV is missing columns: {absent}')
    frame['_doi'] = frame['DOI'].map(normalize_doi)
    if frame['_doi'].eq('').any() or frame['_doi'].duplicated().any():
        raise ValueError(f'{kind} contains blank or duplicate DOI values. Resolve these before scoring.')
    return frame, raw, {'source': source, 'sha256': sha256_bytes(raw), 'rows': len(frame)}

human, human_bytes, human_info = load_csv_bytes('human', HUMAN_CSV)
llm, llm_bytes, llm_info = load_csv_bytes('llm', LLM_CSV)
INPUTS_MATCH_PUBLISHED = (human_info['sha256'] == PUBLISHED_INPUT_SHA256['human']
                          and llm_info['sha256'] == PUBLISHED_INPUT_SHA256['llm'])
if set(human['_doi']) != set(llm['_doi']):
    raise ValueError('The DOI sets differ. HUM-only: ' + str(sorted(set(human['_doi'])-set(llm['_doi'])))
                     + '; LLM-only: ' + str(sorted(set(llm['_doi'])-set(human['_doi']))))
if len(human) != EXPECTED_INPUT_PAPERS or len(llm) != EXPECTED_INPUT_PAPERS:
    raise ValueError(f'Expected {EXPECTED_INPUT_PAPERS} records per input; got {len(human)} and {len(llm)}.')
missing_exclusions = set(EXCLUSIONS) - set(human['_doi'])
if missing_exclusions:
    raise ValueError(f'Expected excluded DOI(s) missing from inputs: {sorted(missing_exclusions)}')

joined = human[['_doi', 'Title'] + FIELDS].merge(
    llm[['_doi', 'Title'] + FIELDS], on='_doi', suffixes=('_human', '_llm'), validate='one_to_one')
joined = joined.sort_values('_doi').reset_index(drop=True)
exclusion_table = joined.loc[joined['_doi'].isin(EXCLUSIONS), ['_doi', 'Title_human']].copy()
exclusion_table['zotero_key'] = exclusion_table['_doi'].map(lambda d: EXCLUSIONS[d]['zotero_key'])
exclusion_table['reason'] = exclusion_table['_doi'].map(lambda d: EXCLUSIONS[d]['reason'])
analysis_data = joined.loc[~joined['_doi'].isin(EXCLUSIONS)].reset_index(drop=True)
if len(analysis_data) != EXPECTED_ANALYZED_PAPERS:
    raise ValueError(f'Expected {EXPECTED_ANALYZED_PAPERS} retained papers; got {len(analysis_data)}.')

PACKAGES = {name: importlib.metadata.version(name) for name in
            ['bert-score', 'transformers', 'torch', 'numpy', 'pandas', 'scipy', 'nltk', 'matplotlib']}
ACTIVE_FIELDS = FIELDS if RUN_MODE == 'full' else ['Objective', 'Scalability', 'LULC Classes']
REASSIGNMENTS = N_SHUFFLES if RUN_MODE == 'full' else 2
CONFIG = {
    'analysis_version': ANALYSIS_VERSION, 'source_commit': SOURCE_COMMIT,
    'run_mode': RUN_MODE, 'fields': ACTIVE_FIELDS, 'n_shuffles': REASSIGNMENTS,
    'seed': RANDOM_SEED, 'include_pairwise_set_f1': INCLUDE_PAIRWISE_SET_F1,
    'model_type': MODEL_TYPE, 'num_layers': MODEL_LAYERS, 'idf': False,
    'rescale_with_baseline': False, 'use_fast_tokenizer': False,
    'pair_threshold': PAIR_THRESHOLD, 'exclusions': EXCLUSIONS,
    'historical_mean_tolerance': HISTORICAL_MEAN_TOLERANCE,
    'inputs': {'human': human_info, 'llm': llm_info}, 'packages': PACKAGES,
    'requested_device': DEVICE, 'batch_size': BATCH_SIZE, 'score_chunk_size': SCORE_CHUNK_SIZE,
}
RUN_ID = RUN_MODE + '_' + digest_json(CONFIG)[:12]
RUN_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'inputs').mkdir(exist_ok=True)
(RUN_DIR / 'inputs' / 'human.csv').write_bytes(human_bytes)
(RUN_DIR / 'inputs' / 'llm.csv').write_bytes(llm_bytes)
write_csv(exclusion_table.rename(columns={'_doi': 'doi', 'Title_human': 'title'}), 'excluded_papers.csv')
write_csv(analysis_data[['_doi', 'Title_human', 'Title_llm']].rename(columns={'_doi': 'doi'}),
          'included_papers.csv')
MANIFEST = {**CONFIG, 'started_utc': datetime.now(timezone.utc).isoformat(),
            'python': sys.version, 'platform': platform.platform(),
            'analyzed_papers': len(analysis_data), 'completed': False,
            'historical_environment_fully_known': False,
            'interpretation': 'Descriptive within-corpus similarity calibration, not factual accuracy.'}
write_json(RUN_DIR / 'run_manifest.json', MANIFEST)
historical_summary = pd.DataFrame(HISTORICAL_SUMMARY_ROWS)

print(f'Input records: {len(joined)} | Original exclusions: {len(exclusion_table)} | Retained: {len(analysis_data)}')
print('Output folder:', RUN_DIR.resolve())
print('Mode:', RUN_MODE.upper(), '| Reassignments:', REASSIGNMENTS)
display(exclusion_table.rename(columns={'_doi': 'DOI', 'Title_human': 'Title'}))


## 4. Prepare text and identify missing fields

The primary baseline uses only records with substantive text on **both** sides for a given
field. Whole-cell placeholders such as “Not specified” are excluded. Other statements,
including substantive negative findings, are retained. A field-level audit records all
exclusions, and the same eligible paper set is used for matched and mismatched comparisons.

For comparability, preprocessing mirrors the repository: narrative text is lowercased
and punctuation removed. Set fields use its separators, parenthesis removal, noun
lemmatization, deduplication, and exact filler-token removal before text scoring.

Separately, the historical-method check retains literal “Not specified” values, as the
original evaluator did. Those scores are compared with the cached means; they are not
substituted for the substantive-text baseline. This makes any change of denominator visible.


In [ ]:
# Ported from the recorded evaluator; no imports of the full legacy application are needed.
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    try:
        nltk.data.find('corpora/wordnet.zip')
    except LookupError:
        if not nltk.download('wordnet', quiet=False):
            raise RuntimeError('WordNet download failed. Check internet access and rerun this cell.')
LEMMATIZER = WordNetLemmatizer()
LEMMATIZER.lemmatize('models')  # Check that the resource is usable before preparing fields.
SEPARATORS = (',', ';', '/', ' and ', ' & ', '、', '，', '|')
SET_SPLIT = re.compile('(?:' + '|'.join(re.escape(s) for s in SEPARATORS) + ')')

def preprocess_text(value):
    if pd.isna(value):
        return ''
    return re.sub(r'[^\w\s]', '', str(value).lower()).strip()

def normalize_unicode_whitespace(value):
    text = unicodedata.normalize('NFKC', str(value))
    text = text.replace('–', '-').replace('—', '-').replace('−', '-')
    text = text.replace('•', '; ').replace('·', '; ').replace('‧', '; ')
    return re.sub(r'\s+', ' ', text).strip()

def strip_parentheses(text):
    return re.sub(r'\([^)]*\)', ' ', text)

def clean_set_cell(value):
    if pd.isna(value):
        return ''
    text = strip_parentheses(normalize_unicode_whitespace(value))
    items = []
    for item in SET_SPLIT.split(text):
        item = item.strip()
        if item:
            items.append(' '.join(LEMMATIZER.lemmatize(word) for word in item.lower().split()))
    ignore = {preprocess_text(s) for s in ['model', 'models', 'framework', 'approach']}
    items = [item for item in items if preprocess_text(item) not in ignore]
    return '; '.join(dict.fromkeys(items)).strip()

def split_set_items(cleaned):
    text = strip_parentheses(normalize_unicode_whitespace(cleaned))
    return [s.strip() for s in SET_SPLIT.split(text) if s.strip()]

PLACEHOLDERS = {
    '', 'not specified', 'not reported', 'not mentioned', 'not available', 'not applicable',
    'not explicitly specified', 'not explicitly reported', 'not explicitly mentioned',
    'not explicitly stated', 'unspecified', 'none', 'nan', 'null', 'na', 'n a',
}

def is_placeholder(value):
    if pd.isna(value):
        return True
    normalized = re.sub(r'\s+', ' ', preprocess_text(normalize_unicode_whitespace(value)))
    return normalized in PLACEHOLDERS

def legacy_missing(value):
    # The original evaluator did NOT treat 'Not specified' as missing.
    return pd.isna(value) or str(value).strip() in ['', 'nan', 'NaN', 'None', 'null']

def stable_seed(name):
    return (RANDOM_SEED + int(hashlib.sha256(name.encode('utf-8')).hexdigest()[:8], 16)) % (2**32)

prepared_records = []
for field in ACTIVE_FIELDS:
    kind = 'set_based' if field in SET_FIELDS else 'free_text'
    for _, row in analysis_data.iterrows():
        h_raw, l_raw = row[field + '_human'], row[field + '_llm']
        h_legacy = clean_set_cell(h_raw) if kind == 'set_based' else h_raw
        l_legacy = clean_set_cell(l_raw) if kind == 'set_based' else l_raw
        h_items = split_set_items(h_legacy) if kind == 'set_based' else []
        l_items = split_set_items(l_legacy) if kind == 'set_based' else []
        h_text = preprocess_text('; '.join(h_items) if kind == 'set_based' else h_legacy)
        l_text = preprocess_text('; '.join(l_items) if kind == 'set_based' else l_legacy)
        h_missing = is_placeholder(h_raw) or is_placeholder(h_text)
        l_missing = is_placeholder(l_raw) or is_placeholder(l_text)
        reason = 'both_missing' if h_missing and l_missing else (
                 'human_missing' if h_missing else ('llm_missing' if l_missing else 'eligible'))
        prepared_records.append({
            'field': field, 'field_type': kind, 'doi': row['_doi'], 'title': row['Title_human'],
            'human_raw': h_raw, 'llm_raw': l_raw, 'human_text': h_text, 'llm_text': l_text,
            'human_items': json.dumps(h_items, ensure_ascii=False),
            'llm_items': json.dumps(l_items, ensure_ascii=False),
            'human_placeholder': h_missing, 'llm_placeholder': l_missing,
            'eligibility': reason, 'eligible': reason == 'eligible',
            'legacy_valid': not legacy_missing(h_legacy) and not legacy_missing(l_legacy),
        })
prepared = pd.DataFrame(prepared_records)
field_data = {}
audit_records = []
for field in ACTIVE_FIELDS:
    sub = prepared.loc[prepared.field.eq(field)].copy()
    eligible = sub.loc[sub.eligible].sort_values('doi').reset_index(drop=True)
    if RUN_MODE == 'quick' and len(eligible) > 12:
        eligible = eligible.sample(12, random_state=stable_seed(field)).sort_values('doi').reset_index(drop=True)
    if len(eligible) < 2:
        raise ValueError(f'{field}: fewer than two eligible papers; a mismatch baseline cannot be calculated.')
    field_data[field] = eligible
    counts = sub.eligibility.value_counts()
    audit_records.append({
        'field': field, 'field_type': sub.field_type.iloc[0], 'retained_papers': len(sub),
        'both_missing': int(counts.get('both_missing', 0)),
        'human_only_missing': int(counts.get('human_missing', 0)),
        'llm_only_missing': int(counts.get('llm_missing', 0)),
        'substantive_pairs_available': int(sub.eligible.sum()), 'pairs_analyzed': len(eligible),
        'legacy_pairs_available': int(sub.legacy_valid.sum()), 'run_mode': RUN_MODE,
    })
field_audit = pd.DataFrame(audit_records)
write_csv(prepared, 'prepared_fields.csv')
write_csv(field_audit, 'field_eligibility_audit.csv')
write_json(RUN_DIR / 'missing_value_rules.json', {
    'whole_cell_placeholders_after_normalization': sorted(PLACEHOLDERS),
    'primary_rule': 'Both sides must contain substantive text, before and after preprocessing.',
    'legacy_rule': "Literal 'Not specified' retained; original blank-value rules used.",
})
display(field_audit)


## 5. Load the scorer and audit token truncation

The public RoBERTa model runs locally. Downloading it does not send your extraction texts
to an inference API. The model's token limit is inherited from BERTScore; texts over that
limit are explicitly listed in `token_length_audit.csv`. The notebook does not change
the original scoring procedure by chunking or summarizing long entries.


In [ ]:
resolved_device = ('cuda' if torch.cuda.is_available() else 'cpu') if DEVICE == 'auto' else DEVICE
if resolved_device == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available in this kernel. Set DEVICE="cpu" or "auto".')
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
print('Loading', MODEL_TYPE, '| layer', MODEL_LAYERS, '| device', resolved_device)
scorer = BERTScorer(
    model_type=MODEL_TYPE, num_layers=MODEL_LAYERS, lang='en',
    idf=False, rescale_with_baseline=False, use_fast_tokenizer=False,
    batch_size=BATCH_SIZE, nthreads=min(4, os.cpu_count() or 1), device=resolved_device,
)
model_revision = getattr(scorer._model.config, '_commit_hash', None)
MODEL_INFO = {
    'requested_model': MODEL_TYPE, 'num_layers': MODEL_LAYERS,
    'bertscore_hash': scorer.hash, 'resolved_revision': model_revision,
    'revision_known': model_revision is not None, 'device': resolved_device,
    'model_config': scorer._model.config.to_dict(),
    'tokenizer_class': type(scorer._tokenizer).__name__,
    'tokenizer_init': {k: str(v) for k, v in scorer._tokenizer.init_kwargs.items()
                       if k in ['name_or_path', 'revision', '_commit_hash', 'model_max_length']},
    'packages': PACKAGES,
}
SCORE_FINGERPRINT = digest_json({
    'analysis_version': ANALYSIS_VERSION, 'model_info': MODEL_INFO,
    'idf': False, 'rescale_with_baseline': False, 'use_fast_tokenizer': False,
    'batch_size': BATCH_SIZE, 'score_chunk_size': SCORE_CHUNK_SIZE,
    # If the resolved weights cannot be identified, do not reuse scores across runs.
    'unknown_revision_session': MANIFEST['started_utc'] if model_revision is None else None,
})
MANIFEST['model'] = MODEL_INFO
MANIFEST['score_fingerprint'] = SCORE_FINGERPRINT
write_json(RUN_DIR / 'run_manifest.json', MANIFEST)
print('Scorer:', scorer.hash)
print('Resolved model revision:', model_revision or 'not exposed by this environment')

# BERTScore truncates through its tokenizer. Record original lengths for audit.
token_limit = int(scorer._tokenizer.model_max_length)
if token_limit > 100_000:
    token_limit = int(getattr(scorer._model.config, 'max_position_embeddings', 514)) - 2
token_cache = {}

def token_length(text):
    if text not in token_cache:
        extra = {'add_prefix_space': True} if type(scorer._tokenizer).__name__ in {
            'RobertaTokenizer', 'GPT2Tokenizer'} else {}
        token_cache[text] = len(scorer._tokenizer.encode(
            text.strip(), add_special_tokens=True, truncation=False, **extra))
    return token_cache[text]

token_rows = []
for _, row in prepared.iterrows():
    for side in ['human', 'llm']:
        text = row[side + '_text']
        length = token_length(text)
        token_rows.append({'field': row.field, 'doi': row.doi, 'side': side,
                           'tokens_before_truncation': length, 'token_limit': token_limit,
                           'exceeds_limit': length > token_limit, 'primary_eligible': bool(row.eligible)})
token_audit = pd.DataFrame(token_rows)
write_csv(token_audit, 'token_length_audit.csv')
print('Field entries exceeding the token limit:', int(token_audit.exceeds_limit.sum()))
if token_audit.exceeds_limit.any():
    display(token_audit.loc[token_audit.exceeds_limit])

# Cache key includes scorer/environment information and the ordered text pair.
CACHE_PATH = OUTPUT_ROOT / 'score_cache.sqlite'
cache = sqlite3.connect(CACHE_PATH)
cache.execute('CREATE TABLE IF NOT EXISTS scores (key TEXT PRIMARY KEY, f1 REAL NOT NULL)')
cache.commit()

def pair_key(candidate, reference):
    return digest_json([SCORE_FINGERPRINT, candidate, reference])

def score_text_pairs(candidates, references, label=''):
    if len(candidates) != len(references):
        raise ValueError('Candidate and reference counts differ.')
    if not candidates:
        return np.empty(0, dtype=float)
    pairs = list(zip(candidates, references))
    keys = [pair_key(c, r) for c, r in pairs]
    unique = dict(zip(keys, pairs))
    cached = {}
    all_keys = list(unique)
    for start in range(0, len(all_keys), 400):
        chunk = all_keys[start:start+400]
        sql = 'SELECT key,f1 FROM scores WHERE key IN (' + ','.join('?' for _ in chunk) + ')'
        cached.update(cache.execute(sql, chunk).fetchall())
    pending = [(key, c, r) for key, (c, r) in unique.items() if key not in cached]
    print(f'{label}: {len(pairs):,} pairs; {len(unique):,} unique; {len(pending):,} to calculate', flush=True)
    for start in range(0, len(pending), SCORE_CHUNK_SIZE):
        chunk = pending[start:start+SCORE_CHUNK_SIZE]
        with torch.inference_mode():
            _, _, tensor = scorer.score([x[1] for x in chunk], [x[2] for x in chunk],
                                         batch_size=BATCH_SIZE, verbose=True)
        values = tensor.detach().cpu().numpy().astype(float)
        if len(values) != len(chunk) or not np.isfinite(values).all():
            raise RuntimeError('Scorer returned missing/nonfinite values. No replacement values were inserted.')
        new = [(x[0], float(value)) for x, value in zip(chunk, values)]
        cache.executemany('INSERT OR REPLACE INTO scores (key,f1) VALUES (?,?)', new)
        cache.commit()
        cached.update(new)
        print(f'  Saved {min(start+len(chunk), len(pending)):,}/{len(pending):,} new scores', flush=True)
    return np.array([cached[key] for key in keys], dtype=float)


## 6. Create the mismatched pairs

Each reassignment is a random permutation with **no fixed points**: every LLM entry is
paired with a human entry from a different DOI, and each eligible human entry is used once
per reassignment. Rejection sampling creates these permutations without deliberately
selecting similar or dissimilar papers. Different fields can have different eligible sets.
All pairings are saved before scoring. Identical wording across different papers is allowed
and counted in the audit, because shared wording is part of the baseline being assessed.


In [ ]:
def random_derangement(n, rng):
    if n < 2:
        raise ValueError('At least two papers are required.')
    positions = np.arange(n)
    for _ in range(10000):
        shuffled = rng.permutation(n)
        if np.all(shuffled != positions):
            return shuffled
    raise RuntimeError('Could not generate a derangement; no biased fallback was used.')

pair_plans = {}
pairing_records = []
for field in ACTIVE_FIELDS:
    sub = field_data[field]
    rng = np.random.default_rng(stable_seed(field))
    indices = np.arange(len(sub))
    mappings = [indices] + [random_derangement(len(sub), rng) for _ in range(REASSIGNMENTS)]
    for repetition, mapping in enumerate(mappings):
        assert len(set(mapping)) == len(sub)
        if repetition:
            assert np.all(mapping != indices)
        for i, j in enumerate(mapping):
            same_doi = sub.iloc[i].doi == sub.iloc[j].doi
            assert same_doi == (repetition == 0)
            pairing_records.append({
                'field': field, 'pair_type': 'matched' if repetition == 0 else 'mismatched',
                'reassignment': repetition, 'llm_index': i, 'human_index': int(j),
                'llm_doi': sub.iloc[i].doi, 'human_doi': sub.iloc[j].doi,
                'same_doi': same_doi,
                'identical_preprocessed_text': sub.iloc[i].llm_text == sub.iloc[j].human_text,
            })
    pair_plans[field] = mappings
pairing_audit = pd.DataFrame(pairing_records)
write_csv(pairing_audit, 'pairing_audit.csv')
print('Planned direct BERT-F1 comparisons:', f'{len(pairing_audit):,}')
print('Incorrect same-paper matches in baseline:', int(
    pairing_audit.loc[pairing_audit.pair_type.eq('mismatched'), 'same_doi'].sum()))
print('Mismatched pairs with identical preprocessed text:', int(
    pairing_audit.loc[pairing_audit.pair_type.eq('mismatched'), 'identical_preprocessed_text'].sum()))
if INCLUDE_PAIRWISE_SET_F1:
    print('The optional set-item extension will require additional item-pair comparisons.')


## 7. Run the analysis

This is the main computation. It scores the matched and mismatched pairs under identical
settings and saves completed scores to a local SQLite cache. If execution is interrupted,
run the notebook again with the same settings to reuse cached pairs. Progress messages
identify the field and number of remaining comparisons.

The optional set-item extension uses the original Hungarian assignment followed by a
0.70 BERT-F1 threshold, and is labeled `thresholded_set_pair_f1`. It is not a mean of token
BERT-F1 values. The main analysis does not need this extension to run.


In [ ]:
ANALYSIS_FINISHED = False
all_summaries, all_reassignments, all_papers, all_pair_scores, all_historical = [], [], [], [], []
analysis_started = time.perf_counter()

def save_partial():
    write_csv(pd.DataFrame(all_summaries), 'baseline_summary.partial.csv')
    write_csv(pd.DataFrame(all_historical), 'historical_method_check.partial.csv')

def record_primary(field, metric, scores, sub):
    n = len(sub)
    matrix = np.asarray(scores, dtype=float).reshape(REASSIGNMENTS + 1, n)
    if not np.isfinite(matrix).all():
        raise ValueError(f'{field}/{metric}: nonfinite scores.')
    matched, mismatched = matrix[0], matrix[1:]
    shuffle_means = mismatched.mean(axis=1)
    all_summaries.append({
        'field': field, 'field_type': sub.field_type.iloc[0], 'metric': metric,
        'n_papers': n, 'n_reassignments': REASSIGNMENTS,
        'matched_mean': float(matched.mean()), 'mismatched_mean': float(mismatched.mean()),
        'matched_minus_mismatched': float(matched.mean() - mismatched.mean()),
        'reassignment_mean_sd': float(shuffle_means.std(ddof=1)),
        'reassignment_mean_min': float(shuffle_means.min()),
        'reassignment_mean_max': float(shuffle_means.max()),
        'denominator': 'Both human and LLM entries contain substantive text', 'run_mode': RUN_MODE,
    })
    for repetition, mean in enumerate(shuffle_means, start=1):
        all_reassignments.append({'field': field, 'metric': metric, 'reassignment': repetition,
                                  'n_papers': n, 'mismatched_mean': float(mean),
                                  'matched_mean': float(matched.mean()),
                                  'matched_minus_mismatched': float(matched.mean() - mean)})
    for i, row in sub.iterrows():
        all_papers.append({'field': field, 'metric': metric, 'doi': row.doi, 'title': row.title,
                           'matched_f1': float(matched[i]),
                           'mean_mismatched_f1': float(mismatched[:, i].mean()),
                           'difference': float(matched[i] - mismatched[:, i].mean()),
                           'n_reassignments': REASSIGNMENTS})
    pairs = pairing_audit.loc[pairing_audit.field.eq(field)].copy().reset_index(drop=True)
    assert len(pairs) == len(scores)
    pairs['metric'] = metric
    pairs['score'] = scores
    all_pair_scores.append(pairs)

def record_historical(field, metric, scores, legacy):
    cached_metric = {'bert_f1': 'bert_f1', 'set_text_bert_f1': 'set_text_bert_f1',
                     'thresholded_set_pair_f1': 'soft_f1'}[metric]
    cached_row = historical_summary.loc[
        historical_summary.Column.eq(field) & historical_summary.Metric.eq(cached_metric)]
    historical_mean = float(cached_row.Mean.iloc[0]) if len(cached_row) == 1 else np.nan
    current_mean = float(np.mean(scores)) if len(scores) else np.nan
    comparable = RUN_MODE == 'full' and INPUTS_MATCH_PUBLISHED and len(legacy) == EXPECTED_ANALYZED_PAPERS
    gap = current_mean - historical_mean if comparable else np.nan
    status = ('check_difference' if abs(gap) > HISTORICAL_MEAN_TOLERANCE else 'within_tolerance') if comparable else 'not_directly_comparable'
    all_historical.append({
        'field': field, 'metric': metric, 'cached_metric_name': cached_metric,
        'recomputed_original_method_mean': current_mean, 'cached_original_mean': historical_mean,
        'difference_from_cached_mean': gap, 'n_recomputed': len(legacy),
        'full_original_cohort_comparison': comparable, 'status': status,
        'absolute_tolerance': HISTORICAL_MEAN_TOLERANCE,
        'note': "Original-style calculation retains literal 'Not specified' entries; this is separate from the substantive-text baseline.",
    })

def threshold_pair_scores(document_pairs, label):
    # Match the original algorithm: maximize total BERT-F1, then threshold at 0.70.
    # This is NOT a mean of item similarities or a token-level BERT-F1 score.
    result = np.empty(len(document_pairs), dtype=float)
    for start in range(0, len(document_pairs), 64):
        docs = document_pairs[start:start+64]
        candidates, references, layouts = [], [], []
        for local, (llm_items, human_items) in enumerate(docs):
            a, b = len(llm_items), len(human_items)
            if not a or not b:
                result[start+local] = 1.0 if a == b == 0 else 0.0
                continue
            offset = len(candidates)
            for llm_item in llm_items:
                for human_item in human_items:
                    candidates.append(preprocess_text(llm_item))
                    references.append(preprocess_text(human_item))
            layouts.append((local, offset, a, b))
        values = score_text_pairs(candidates, references,
                                  f'{label} documents {start+1}-{start+len(docs)}')
        for local, offset, a, b in layouts:
            similarity = values[offset:offset+a*b].reshape(a, b)
            ii, jj = linear_sum_assignment(-similarity)
            matched_count = int(np.sum(similarity[ii, jj] >= PAIR_THRESHOLD))
            result[start+local] = 2.0 * matched_count / (a + b)
    return result

for field_number, field in enumerate(ACTIVE_FIELDS, start=1):
    started = time.perf_counter()
    sub = field_data[field]
    mappings = pair_plans[field]
    metric = 'set_text_bert_f1' if field in SET_FIELDS else 'bert_f1'
    print(f'\nFIELD {field_number}/{len(ACTIVE_FIELDS)}: {field} ({len(sub)} substantive pairs)', flush=True)
    candidates, references = [], []
    for mapping in mappings:
        candidates.extend(sub.llm_text.tolist())
        references.extend(sub.iloc[mapping].human_text.tolist())
    primary_count = len(candidates)
    legacy = prepared.loc[prepared.field.eq(field) & prepared.legacy_valid].sort_values('doi')
    if RUN_MODE == 'quick':
        legacy = legacy.loc[legacy.doi.isin(sub.doi)]
    candidates.extend(legacy.llm_text.tolist())
    references.extend(legacy.human_text.tolist())
    values = score_text_pairs(candidates, references, field)
    record_primary(field, metric, values[:primary_count], sub)
    legacy_values = values[primary_count:].copy()
    # Preserve the legacy text_similarity both-empty convention only in the historical check.
    for i, (_, row) in enumerate(legacy.iterrows()):
        if not row.llm_text and not row.human_text:
            legacy_values[i] = 1.0
    record_historical(field, metric, legacy_values, legacy)

    if INCLUDE_PAIRWISE_SET_F1 and field in SET_FIELDS:
        llm_items = [json.loads(value) for value in sub.llm_items]
        human_items = [json.loads(value) for value in sub.human_items]
        document_pairs = [(llm_items[i], human_items[j]) for mapping in mappings
                          for i, j in enumerate(mapping)]
        document_pairs.extend((json.loads(row.llm_items), json.loads(row.human_items))
                              for _, row in legacy.iterrows())
        item_values = threshold_pair_scores(document_pairs, field + ' thresholded set F1')
        record_primary(field, 'thresholded_set_pair_f1', item_values[:primary_count], sub)
        record_historical(field, 'thresholded_set_pair_f1', item_values[primary_count:], legacy)
    save_partial()
    print(f'{field} finished in {(time.perf_counter()-started)/60:.1f} minutes; progress saved.', flush=True)

baseline_summary = pd.DataFrame(all_summaries)
baseline_by_reassignment = pd.DataFrame(all_reassignments)
per_paper_comparison = pd.DataFrame(all_papers)
pair_scores = pd.concat(all_pair_scores, ignore_index=True)
historical_check = pd.DataFrame(all_historical)
expected_metrics = len(ACTIVE_FIELDS) + (sum(f in SET_FIELDS for f in ACTIVE_FIELDS) if INCLUDE_PAIRWISE_SET_F1 else 0)
assert len(baseline_summary) == expected_metrics
assert not baseline_summary.duplicated(['field', 'metric']).any()
assert np.isfinite(baseline_summary[['matched_mean', 'mismatched_mean', 'matched_minus_mismatched']]).all().all()
assert not pair_scores.loc[pair_scores.pair_type.eq('mismatched'), 'same_doi'].any()
for _, group in pair_scores.loc[pair_scores.pair_type.eq('mismatched')].groupby(['field', 'metric', 'reassignment']):
    assert group.llm_doi.nunique() == len(group) == group.human_doi.nunique()

for frame, name in [(baseline_summary, 'baseline_summary.csv'),
                    (baseline_by_reassignment, 'baseline_by_reassignment.csv'),
                    (per_paper_comparison, 'per_paper_comparison.csv'),
                    (pair_scores, 'pair_scores.csv'),
                    (historical_check, 'historical_method_check.csv')]:
    write_csv(frame, name)
ANALYSIS_FINISHED = True
print(f'\nScoring complete in {(time.perf_counter()-analysis_started)/60:.1f} minutes.')


## 8. Inspect and export results

- `baseline_summary.csv`: primary matched mean, mismatched mean, and difference by field.
- `baseline_by_reassignment.csv`: each reassignment's mean; its range/SD describes
  reassignment variability, **not** a confidence interval for generalization to other papers.
- `per_paper_comparison.csv` and `pair_scores.csv`: detailed numerical results.
- `historical_method_check.csv`: recomputed original-style matched means versus saved means.
- `field_eligibility_audit.csv`, `prepared_fields.csv`, `pairing_audit.csv`, and
  `token_length_audit.csv`: denominators, text handling, pair assignments, and truncation.
- `matched_vs_mismatched.png` / `.pdf`: descriptive figure with no significance claims.
- `run_manifest.json`: configuration, input hashes, model information, versions, and completion.

The repeated reassignments are not additional independent papers. No significance test,
accuracy percentage, or factual-completeness claim is inferred from BERTScore. Inspect small
or negative differences as carefully as positive differences. A separate shortfall in
extraction completeness remains relevant even when this conditional text-similarity gap is large.


In [ ]:
if not globals().get('ANALYSIS_FINISHED', False):
    raise RuntimeError('The analysis cell has not finished. Run it before exporting results.')

print('RESULTS —', RUN_MODE.upper())
if RUN_MODE == 'quick':
    print('QUICK CHECK ONLY — do not use these results in the manuscript.')
display(baseline_summary[['field', 'metric', 'n_papers', 'matched_mean',
                          'mismatched_mean', 'matched_minus_mismatched',
                          'reassignment_mean_min', 'reassignment_mean_max']].round(4))

print('\nCheck against the original saved means (different from the substantive-text analysis):')
display(historical_check[['field', 'metric', 'n_recomputed', 'recomputed_original_method_mean',
                          'cached_original_mean', 'difference_from_cached_mean', 'status']].round(5))
check_fields = historical_check.loc[historical_check.status.eq('check_difference'), 'field'].unique().tolist()
if check_fields:
    print('Investigate these historical mean differences before replacing original manuscript scores:', check_fields)

plot_data = baseline_summary.loc[baseline_summary.metric.isin(['bert_f1', 'set_text_bert_f1'])]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, layout='constrained')
for ax, kind, title in zip(axes, ['free_text', 'set_based'], ['Narrative fields', 'Whole-set BERTScore']):
    sub = plot_data.loc[plot_data.field_type.eq(kind)].reset_index(drop=True)
    y = np.arange(len(sub))
    for i, row in sub.iterrows():
        ax.plot([row.mismatched_mean, row.matched_mean], [i, i], color='#A9A9A9', linewidth=1.5, zorder=1)
    ax.scatter(sub.matched_mean, y, color='#176B87', s=44, label='Matched papers', zorder=3)
    ax.scatter(sub.mismatched_mean, y, color='#C56A2D', marker='s', s=36, label='Mismatched papers', zorder=3)
    ax.set_yticks(y, sub.field)
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel('Mean BERT-F1')
    ax.set_title(title)
    ax.grid(axis='x', alpha=0.20)
    ax.spines[['top', 'right']].set_visible(False)
axes[0].legend(loc='lower left', frameon=False, fontsize=9)
fig.suptitle('Matched and mismatched extraction similarity' + (' — QUICK CHECK' if RUN_MODE == 'quick' else ''), fontsize=14)
fig.savefig(RUN_DIR / 'matched_vs_mismatched.png', dpi=220, bbox_inches='tight')
fig.savefig(RUN_DIR / 'matched_vs_mismatched.pdf', bbox_inches='tight')
plt.show()

notes = (
    'BERTScore mismatched-paper baseline (Supplementary Note 6)\n\n'
    f'Mode: {RUN_MODE}; retained sample: {len(analysis_data)} papers; reassignments per field: {REASSIGNMENTS}.\n'
    'Primary denominators contain substantive text on both sides and vary by field.\n'
    'Each mismatched reassignment uses a different DOI and uses each eligible reference once.\n'
    'The baseline is same-field, within-corpus background similarity; it is not a factual-accuracy baseline.\n'
    'Matched-minus-mismatched is a difference in similarity, not a percentage improvement in accuracy.\n'
    'Reassignment SD/min/max describe reassignment variability, not confidence intervals.\n'
    'Repeated pairings are not additional independent papers. No significance test is claimed.\n'
    'The historical-method check retains literal placeholders and uses the original-style denominator.\n'
    'Historical package versions are not fully known. Check differences before updating old manuscript numbers.\n'
    f'Thresholded set-item F1 extension included: {INCLUDE_PAIRWISE_SET_F1}.\n'
    f'Scorer: {scorer.hash}.\n'
    'All configuration, source hashes, versions, and completion information are in run_manifest.json.\n'
    'The original CSVs are preserved. No new LLM extraction calls were made.\n'
)
(RUN_DIR / 'RESULTS_README.txt').write_text(notes, encoding='utf-8')
share_names = [
    'baseline_summary.csv', 'baseline_by_reassignment.csv', 'per_paper_comparison.csv',
    'pair_scores.csv', 'historical_method_check.csv', 'field_eligibility_audit.csv',
    'prepared_fields.csv', 'pairing_audit.csv', 'token_length_audit.csv',
    'excluded_papers.csv', 'included_papers.csv', 'missing_value_rules.json',
    'matched_vs_mismatched.png', 'matched_vs_mismatched.pdf', 'RESULTS_README.txt', 'run_manifest.json',
]
MANIFEST.update({
    'completed': True, 'completed_utc': datetime.now(timezone.utc).isoformat(),
    'scoring_minutes': (time.perf_counter()-analysis_started)/60,
    'field_metric_rows': len(baseline_summary), 'scored_document_pairs': len(pair_scores),
    'historical_comparison_fields_to_check': check_fields,
    'token_truncation_entries': int(token_audit.exceeds_limit.sum()),
    'primary_denominators': field_audit[['field', 'pairs_analyzed']].to_dict('records'),
    'share_files': share_names,
})
write_json(RUN_DIR / 'run_manifest.json', MANIFEST)
RESULTS_ZIP = OUTPUT_ROOT / (RUN_ID + '_results.zip')
with zipfile.ZipFile(RESULTS_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for name in share_names:
        path = RUN_DIR / name
        if not path.is_file():
            raise FileNotFoundError(f'Expected completed output missing: {path}')
        archive.write(path, arcname=name)
print('Output folder:', RUN_DIR.resolve())
print('Results archive:', RESULTS_ZIP.resolve())
display(FileLink(str(RESULTS_ZIP)))
cache.close()


## Notes

The results archive (path above) contains the summary, figure, audits and run metadata;
it omits the score cache and the input snapshots.

If a historical-method mean differs materially from its saved value, inspect that output.
A changed denominator in the substantive-text analysis is reported separately and is not
itself evidence of a scoring error.
